In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch

C:\Users\anshu\cv_dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\anshu\cv_dl_env\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\anshu\cv_dl_env\lib\site-packages\torchvision\transforms\v2\__init__.py:54: UserWar

In [3]:
# Cell 3: Load model and tokenizer
model_name = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)

print(f"✅ Model loaded: {model_name}")
print(f"   Parameters: {model.num_parameters():,}")

C:\Users\anshu\cv_dl_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anshu\.cache\huggingface\hub\models--deepset--roberta-base-squad2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downlo

✅ Model loaded: deepset/roberta-base-squad2
   Parameters: 124,056,578


In [4]:
# Cell 4: Define prediction function
def predict_answer(question, context, top_k=3):
    """
    Predict answer span from context using transformer QA model.
    Returns top-k answer candidates with scores.
    """
    result = qa_pipeline(
        question=question,
        context=context,
        top_k=top_k,
        handle_impossible_answer=True
    )

    print(f"❓ Question : {question}")
    print(f"📄 Context  : {context[:100]}...")
    print("-" * 60)

    for i, ans in enumerate(result):
        print(f"  #{i+1}  Answer     : {ans['answer']}")
        print(f"       Confidence : {ans['score']:.4f} ({ans['score']*100:.1f}%)")
        print(f"       Char Span  : [{ans['start']} → {ans['end']}]")
        print()

    return result

In [5]:
# Cell 5: Example 1 — Science
context1 = """
BERT (Bidirectional Encoder Representations from Transformers) was introduced
by Google in 2018. It uses a masked language modeling objective and next sentence
prediction to pre-train deep bidirectional representations from unlabeled text.
BERT achieved state-of-the-art results on eleven NLP tasks.
"""

question1 = "What year was BERT introduced?"
results1 = predict_answer(question1, context1)

❓ Question : What year was BERT introduced?
📄 Context  : 
BERT (Bidirectional Encoder Representations from Transformers) was introduced
by Google in 2018. It...
------------------------------------------------------------
  #1  Answer     : 2018
       Confidence : 0.9643 (96.4%)
       Char Span  : [92 → 96]

  #2  Answer     : Google in 2018
       Confidence : 0.0014 (0.1%)
       Char Span  : [82 → 96]

  #3  Answer     : in 2018
       Confidence : 0.0010 (0.1%)
       Char Span  : [89 → 96]



In [6]:
# Cell 6: Example 2 — History
context2 = """
The transformer architecture was introduced in the paper 'Attention is All You Need'
by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms to draw
global dependencies between input and output, eliminating recurrence and convolutions.
"""

question2 = "Who introduced the transformer architecture?"
results2 = predict_answer(question2, context2)

❓ Question : Who introduced the transformer architecture?
📄 Context  : 
The transformer architecture was introduced in the paper 'Attention is All You Need'
by Vaswani et ...
------------------------------------------------------------
  #1  Answer     : Vaswani et al.
       Confidence : 0.8682 (86.8%)
       Char Span  : [89 → 103]

  #2  Answer     : Vaswani
       Confidence : 0.0465 (4.6%)
       Char Span  : [89 → 96]

  #3  Answer     : Vaswani et al
       Confidence : 0.0221 (2.2%)
       Char Span  : [89 → 102]



In [7]:
# Cell 7: Token-level inspection (what the model actually sees)
def inspect_tokens(question, context):
    """Show tokenization and input IDs fed into the model."""
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    sep_idx = [i for i, t in enumerate(tokens) if t == "</s>"]

    print("🔍 Token Inspection")
    print(f"   Total tokens : {len(tokens)}")
    print(f"   [CLS] token  : {tokens[0]}")
    print(f"   [SEP] tokens : positions {sep_idx}")
    print(f"\n   Question tokens : {tokens[1:sep_idx[0]]}")
    print(f"   Context tokens  : {tokens[sep_idx[0]+1:sep_idx[1]][:10]}...")

    return inputs

inputs = inspect_tokens(question1, context1)

🔍 Token Inspection
   Total tokens : 81
   [CLS] token  : <s>
   [SEP] tokens : positions [8, 9, 80]

   Question tokens : ['What', 'Ġyear', 'Ġwas', 'ĠB', 'ERT', 'Ġintroduced', '?']
   Context tokens  : []...


In [8]:
# Cell 8: Raw logits — see what the model predicts before softmax
def get_raw_logits(question, context):
    """Run the model manually and inspect start/end logits."""
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits   = outputs.end_logits[0]

    start_idx = torch.argmax(start_logits).item()
    end_idx   = torch.argmax(end_logits).item()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    answer = tokenizer.convert_tokens_to_string(tokens[start_idx:end_idx+1])

    print("📊 Raw Logits Analysis")
    print(f"   Best start token : {start_idx} → '{tokens[start_idx]}'")
    print(f"   Best end token   : {end_idx}   → '{tokens[end_idx]}'")
    print(f"   Extracted answer : '{answer}'")
    print(f"\n   Top-5 start positions: {torch.topk(start_logits, 5).indices.tolist()}")
    print(f"   Top-5 end positions  : {torch.topk(end_logits, 5).indices.tolist()}")

    return start_logits, end_logits

get_raw_logits(question1, context1)

📊 Raw Logits Analysis
   Best start token : 31 → 'Ġ2018'
   Best end token   : 31   → 'Ġ2018'
   Extracted answer : ' 2018'

   Top-5 start positions: [31, 0, 14, 29, 30]
   Top-5 end positions  : [31, 0, 23, 60, 32]


(tensor([ 2.1253, -8.5180, -8.9921, -9.2735, -7.6483, -9.3000, -9.3886, -9.7157,
         -9.5130, -8.6980, -0.7450, -5.6756, -6.4120, -6.4438,  0.1734, -4.9834,
         -5.4545, -6.5457, -3.9542, -6.8048, -5.7891, -7.0215, -5.8132, -2.9828,
         -6.4301, -3.2816, -3.4873, -1.9662, -5.8129,  0.1244, -0.1838,  6.6544,
         -5.2482, -4.0798, -7.0558, -6.1883, -5.2232, -5.9853, -6.8538, -6.9839,
         -8.5687, -5.4107, -6.6433, -5.1570, -6.6130, -7.7636, -7.4471, -5.3824,
         -8.0730, -7.4789, -5.3691, -4.5587, -7.7761, -8.7820, -7.7706, -7.8530,
         -5.2810, -7.8575, -8.1615, -5.7156, -5.2482, -1.2834, -6.5055, -7.7245,
         -6.7657, -6.7069, -8.5817, -8.3761, -8.6770, -8.2724, -8.8439, -8.3814,
         -7.8491, -8.7913, -6.4281, -7.0733, -8.7178, -8.0032, -5.2483, -4.7995,
         -9.1457]),
 tensor([ 2.5917, -8.4607, -6.9083, -8.2744, -8.7987, -6.6884, -7.4893, -6.8974,
         -7.3431, -6.3453, -3.8287, -5.0572, -1.1685, -6.4890, -5.4893, -6.6795,
        

In [9]:
# Cell 9: Batch prediction on multiple QA pairs
qa_pairs = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Western Europe. Its capital city is Paris, which is known for the Eiffel Tower."
    },
    {
        "question": "Who wrote the theory of relativity?",
        "context": "Albert Einstein developed the theory of relativity in the early 20th century, fundamentally changing our understanding of space and time."
    },
    {
        "question": "What language is PyTorch written in?",
        "context": "PyTorch is an open-source machine learning framework developed by Meta AI. It is primarily written in Python and C++."
    }
]

print("🚀 Batch QA Prediction\n" + "="*60)
for i, pair in enumerate(qa_pairs):
    result = qa_pipeline(pair["question"], pair["context"])
    print(f"[{i+1}] Q: {pair['question']}")
    print(f"     A: {result['answer']}  (confidence: {result['score']:.2%})")
    print()

🚀 Batch QA Prediction


C:\Users\anshu\cv_dl_env\lib\site-packages\transformers\pipelines\question_answering.py:391: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


[1] Q: What is the capital of France?
     A: Paris  (confidence: 98.05%)

[2] Q: Who wrote the theory of relativity?
     A: Albert Einstein  (confidence: 19.24%)

[3] Q: What language is PyTorch written in?
     A: C++.  (confidence: 51.95%)



In [10]:
# Cell 10: Interactive — ask your own question
your_context = """
The Python programming language was created by Guido van Rossum and first released
in 1991. Python emphasizes code readability and uses significant indentation.
It supports multiple programming paradigms including procedural, object-oriented,
and functional programming.
"""

your_question = "Who created Python?"   # ← change this

result = qa_pipeline(your_question, your_context)
print(f"Answer     : {result['answer']}")
print(f"Confidence : {result['score']:.2%}")
print(f"Char span  : [{result['start']} → {result['end']}]")

Answer     : Guido van Rossum
Confidence : 98.95%
Char span  : [48 → 64]
